# SHAP Global Validation & Feature Importance
## CVD Fairness Dissertation — NB1

**Purpose:** Validate the SHAP implementation, compute global feature importance for the full
test set, and produce sex-stratified explainability plots for clinical interpretation.

**Outputs saved to:** `outputs/shap/`

**This notebook:**
- Validates SHAP additivity and shape correctness
- Produces global summary plots (full test set)
- Produces sex-stratified summary and dependence plots
- Saves SHAP arrays for NB2 to load (no recomputation needed)

**Does NOT contain:** local explanations, waterfall plots, counterfactuals, fairness metrics
(those are in NB2 and fairness_evaluation.ipynb)

## 1. Imports

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.inspection import PartialDependenceDisplay

## 2. Paths & Constants

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_PATH   = Path("../models/baseline_models/cardio_xgb_baseline_model.pkl")
TEST_DATA    = Path("../data/test_train_val_sets/cardio_baseline_test.csv")
CONFIG_PATH  = Path("../config/dataset_split_sizes.json")
OUTPUT_SHAP  = Path("../outputs/shap")
OUTPUT_SHAP.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
THRESHOLD    = 0.5
GENDER_COL   = "gender"
FEMALE_VAL   = 0
MALE_VAL     = 1
DROP_COLS    = ["cardio", "stratify"]

EXPECTED_FEATURES = [
    "age_years", "gender", "height", "weight",
    "ap_hi", "ap_lo", "cholesterol", "gluc",
    "smoke", "alco", "active"
]

# Clinical reference thresholds (Ji et al.)
CLINICAL_REFS = {
    "ap_hi": [
        {"x": 110, "color": "red",    "label": "Female risk threshold (Ji et al.)"},
        {"x": 130, "color": "green",  "label": "Male risk threshold (Ji et al.)"},
    ],
    "age_years": [
        {"x": 52,  "color": "orange", "label": "Peri-menopausal transition (Ji et al.)"},
    ],
}

## 3. Notebook Sanity Checks

Validates that the correct model, test set, feature order, sex distribution, and class balance
are all consistent before any analysis runs. All checks raise `AssertionError` on failure so
the notebook halts immediately rather than silently producing wrong results.

In [ ]:
# ── Load data & model ──────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    expected_sizes = json.load(f)

test_df = pd.read_csv(TEST_DATA)
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df["cardio"]

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

# ── Check 1: Test set size ─────────────────────────────────────────────────
assert len(test_df) == expected_sizes["test"], (
    f"Test size mismatch: got {len(test_df)}, expected {expected_sizes['test']}"
)

# ── Check 2: Feature order matches model training ──────────────────────────
assert set(X_test.columns) == set(EXPECTED_FEATURES), (
    f"Feature set mismatch.\n  Got:      {sorted(X_test.columns.tolist())}\n  Expected: {sorted(EXPECTED_FEATURES)}"
)

# Also update the constant to match actual column order in your data
EXPECTED_FEATURES = list(X_test.columns)

# Check 3: Model loads and predicts without error 
test_preds = model.predict(X_test.iloc[:5])
assert len(test_preds) == 5, "Model prediction smoke test failed"

# Check 4: Sex distribution preserved from original split 
female_frac = (test_df[GENDER_COL] == FEMALE_VAL).mean()
assert 0.62 < female_frac < 0.68, (
    f"Sex distribution unexpected: {female_frac:.3f} female (expected ~0.65)"
)

# Check 5: Class balance preserved
cvd_frac = test_df["cardio"].mean()
assert 0.48 < cvd_frac < 0.52, (
    f"Class balance unexpected: {cvd_frac:.3f} CVD positive (expected ~0.50)"
)

# Check 6: No target leakage — cardio not in X_test 
assert "cardio" not in X_test.columns, "Target column 'cardio' found in X_test — data leakage"

# Check 7: Protected attribute present in feature matrix
assert GENDER_COL in X_test.columns, f"'{GENDER_COL}' missing from X_test"

# Check 8: No nulls in test set
null_counts = X_test.isnull().sum()
assert null_counts.sum() == 0, f"Nulls found in X_test:\n{null_counts[null_counts > 0]}"

# Summary 
y_pred = model.predict(X_test)

print("=" * 55)
print("  SANITY CHECKS — ALL PASSED ✓")
print("=" * 55)
print(f"  Test set size      : {len(test_df):,}")
print(f"  Features           : {X_test.shape[1]}")
print(f"  Female fraction    : {female_frac:.3f}")
print(f"  CVD fraction       : {cvd_frac:.3f}")
print(f"  Decision threshold : {THRESHOLD}")
print("=" * 55)

  SANITY CHECKS — ALL PASSED ✓
  Test set size      : 10,227
  Features           : 11
  Female fraction    : 0.650
  CVD fraction       : 0.495
  Decision threshold : 0.5


## 4. Build Sex Masks & Subgroup Splits

Sex masks are defined once here and reused throughout the notebook.
`gender == 0` = female, `gender == 1` = male — consistent with dataset coding.

In [ ]:
female_mask = test_df[GENDER_COL].values == FEMALE_VAL
male_mask   = test_df[GENDER_COL].values == MALE_VAL

X_test_female = X_test[female_mask].copy()
X_test_male   = X_test[male_mask].copy()
y_test_female = y_test.values[female_mask]
y_test_male   = y_test.values[male_mask]

# ── Validation: masks are mutually exclusive and exhaustive 
assert (female_mask & male_mask).sum() == 0, \
    "Overlap between female and male masks"
assert (female_mask | male_mask).sum() == len(test_df), \
    "Masks do not cover full test set"
assert len(X_test_female) + len(X_test_male) == len(X_test), \
    "Subgroup sizes do not sum to test set size"

print(f"Female patients : {female_mask.sum():,}  ({female_mask.mean()*100:.1f}%)")
print(f"Male patients   : {male_mask.sum():,}  ({male_mask.mean()*100:.1f}%)")
print("Sex masks validated")

Female patients : 6,649  (65.0%)
Male patients   : 3,578  (35.0%)
Sex masks validated


## 5. SHAP Explainer Setup & Additivity Validation

`TreeExplainer` reads the XGBoost tree structure directly to compute **exact** Shapley values
rather than approximations. Validation confirms that for every patient, the sum of all SHAP
values plus the baseline exactly reconstructs the model's log-odds output.

A maximum deviation > 1×10⁻⁵ would indicate misconfiguration and halts the notebook.

In [ ]:
# Build explainer
explainer = shap.TreeExplainer(model)

# Compute SHAP for full test set
# Using the Explanation object API (returns values + base_values + data)
sv_full = explainer(X_test)

print(f"SHAP array shape : {sv_full.values.shape}")
print(f"Expected shape   : {X_test.shape}")

SHAP array shape : (10227, 11)
Expected shape   : (10227, 11)


In [ ]:
# ── Check 1: Shape validation ──────────────────────────────────────────────
assert sv_full.values.shape == X_test.shape, (
    f"SHAP shape mismatch: {sv_full.values.shape} vs expected {X_test.shape}"
)

# ── Check 2: Additivity validation ────────────────────────────────────────
# SHAP values are on log-odds scale for XGBoost classifiers
def _logit(p):
    return np.log(p / (1 - p))

pred_probs  = model.predict_proba(X_test)[:, 1]
pred_logits = _logit(pred_probs)

# For the Explanation object, base_values is per-sample (all identical)
shap_sums = sv_full.base_values + sv_full.values.sum(axis=1)
diffs     = np.abs(pred_logits - shap_sums)

ADDITIVITY_TOL = 1e-5
assert diffs.max() < ADDITIVITY_TOL, (
    f"Additivity check FAILED — max deviation {diffs.max():.2e} exceeds tolerance {ADDITIVITY_TOL:.0e}"
)

print("=" * 50)
print("  SHAP VALIDATION — ALL PASSED ✓")
print("=" * 50)
print(f"  Max deviation    : {diffs.max():.2e}  (tolerance {ADDITIVITY_TOL:.0e})")
print(f"  Mean deviation   : {diffs.mean():.2e}")
print(f"  Median deviation : {np.median(diffs):.2e}")
print(f"  SHAP shape       : {sv_full.values.shape} ✓")
print("=" * 50)

## 6. Sex-Stratified SHAP Computation

SHAP values are computed separately for female and male subgroups so that
feature importance and dependence patterns can be compared directly.
Both are also saved to `outputs/shap/` so NB2 can load them without recomputing.

In [ ]:
sv_female = explainer(X_test_female)
sv_male   = explainer(X_test_male)

# ── Validation: subgroup SHAP shapes ──────────────────────────────────────
assert sv_female.values.shape == X_test_female.shape, \
    f"Female SHAP shape mismatch: {sv_female.values.shape}"
assert sv_male.values.shape == X_test_male.shape, \
    f"Male SHAP shape mismatch: {sv_male.values.shape}"
assert sv_female.values.shape[0] + sv_male.values.shape[0] == len(X_test), \
    "Subgroup SHAP row counts do not sum to full test set"

print(f"Female SHAP shape : {sv_female.values.shape} ✓")
print(f"Male SHAP shape   : {sv_male.values.shape} ✓")